# MetaTool Tool-Selection Agent — Colab Runner

End-to-end notebook for the MetaTool benchmark assignment. Runs three open-source LLMs (Gemma 2 2B-it, Llama 3.2 3B-Instruct, Qwen 2.5 7B-Instruct in 4-bit) sequentially on a balanced sample, then computes per-model metrics.

**Hardware:** designed for a free-tier Colab T4 (~15 GB VRAM). Models are loaded and unloaded one at a time.

**Before running:**
1. Runtime → Change runtime type → GPU (T4).
2. You will need a Hugging Face token with access to `meta-llama/Llama-3.2-3B-Instruct` and `google/gemma-2-2b-it` (both are gated; accept the licenses on their HF model pages once).

## 1. Clone project + install dependencies

> **If you previously ran an older version of this notebook that pinned `torch==2.4.0`:** Colab's torch was downgraded and `torchvision`/`torchaudio` are now broken. Reset the runtime first via **Runtime → Disconnect and delete runtime**, then re-run from cell 1.

In [2]:
# Clone the project repo (this notebook lives inside it).
import os, subprocess, sys
REPO_URL = 'https://github.com/jamelski1/hold-the-line.git'
BRANCH = 'claude/metatool-agent-assignment-ImZDo'
WORK_ROOT = '/content/hold-the-line'
PROJECT_DIR = f'{WORK_ROOT}/metatool-agent'

if not os.path.isdir(WORK_ROOT):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, WORK_ROOT], check=True)
else:
    subprocess.run(['git', '-C', WORK_ROOT, 'pull'], check=True)
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

cwd: /content/hold-the-line/metatool-agent


In [3]:
# Install only what Colab is missing or needs a recent version of.
# IMPORTANT: do NOT pin torch / numpy / pandas. Colab ships modern versions
# and downgrading them silently breaks torchvision/torchaudio. We only need
# the four LLM/eval-side packages below.
!pip install -q -U transformers accelerate bitsandbytes rapidfuzz sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.8 MB/s eta 0:00:00


In [4]:
# Hugging Face login (needed for Gemma 2 + Llama 3.2 gated models).
from huggingface_hub import login
from getpass import getpass
import os
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF token: ')
login(os.environ['HF_TOKEN'])

HF token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Download MetaTool data

Clones the MetaTool repo (`HowieHwong/MetaTool`) and copies `big_tool_des.json` and `all_clean_data.csv` into our `data/` directory. The cell prints the schema of each file before any sampling code runs, per the brief.

In [5]:
import os, shutil, subprocess, json, glob
import pandas as pd

DATA_JSON = 'data/big_tool_des.json'
DATA_CSV  = 'data/all_clean_data.csv'

# If the files are already present in the cloned repo, skip the download.
if os.path.isfile(DATA_JSON) and os.path.isfile(DATA_CSV):
    print(f'Found existing data files — skipping MetaTool download.')
else:
    META_REPO = '/content/MetaTool'
    if not os.path.isdir(META_REPO):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/HowieHwong/MetaTool.git', META_REPO], check=True)
    candidates_json = glob.glob(f'{META_REPO}/**/big_tool_des.json', recursive=True)
    candidates_csv  = glob.glob(f'{META_REPO}/**/all_clean_data.csv', recursive=True)
    assert candidates_json, 'big_tool_des.json not found in MetaTool repo'
    assert candidates_csv,  'all_clean_data.csv not found in MetaTool repo'
    src_json, src_csv = candidates_json[0], candidates_csv[0]
    print('Found:', src_json); print('Found:', src_csv)
    os.makedirs('data', exist_ok=True)
    shutil.copy(src_json, DATA_JSON)
    shutil.copy(src_csv,  DATA_CSV)
    print('Copied into data/')

Found existing data files — skipping MetaTool download.


In [6]:
# Inspect schema BEFORE running sampling (per brief).
with open('data/big_tool_des.json') as f:
    raw = json.load(f)
print('big_tool_des.json root type:', type(raw).__name__, '| size:', len(raw))
if isinstance(raw, list):
    print('first entry keys:', list(raw[0].keys()) if isinstance(raw[0], dict) else type(raw[0]))
    print(json.dumps(raw[:2], indent=2)[:600])
elif isinstance(raw, dict):
    keys = list(raw.keys())
    print('first 5 tool names:', keys[:5])
    print('sample value:', json.dumps(raw[keys[0]], indent=2)[:300])

df = pd.read_csv('data/all_clean_data.csv')
print('\nall_clean_data.csv columns:', list(df.columns))
print('shape:', df.shape)
df.head()

big_tool_des.json root type: dict | size: 47
first 5 tool names: ['FinanceTool', 'ExchangeTool', 'NewsTool', 'PolishTool', 'CharityTool']
sample value: "Stay informed with the latest financial updates, real-time insights, and analysis on a wide range of options, stocks, cryptocurrencies, and more."

all_clean_data.csv columns: ['Query', 'Tool']
shape: (20614, 2)


,Query,Tool
0,Can I find academic research papers on this to...,ResearchHelper
1,Can I find any peer-reviewed papers?,ResearchHelper
2,Can I generate bibtex bibliographies?,ResearchHelper
3,Can I use Crossref with Chatbot?,ResearchHelper
4,Can you answer a question about a research pap...,ResearchHelper


## 3. Build sampled dataset

5 positives per tool (~240) + balanced curated negatives. Seeded with `RANDOM_SEED=42`.

In [7]:
from pathlib import Path
from src.sampling import build_sampled_dataset

sampled = build_sampled_dataset(
    big_tool_des_path=Path('data/big_tool_des.json'),
    all_clean_data_path=Path('data/all_clean_data.csv'),
    out_path=Path('data/sampled_data.csv'),
    per_tool=5,
    n_negatives=None,  # balanced
    seed=42,
)
sampled.head()

[sampling] wrote 470 rows (235 positive / 235 negative) -> data/sampled_data.csv


,query,gold_tool,requires_tool
0,Are there any good adventure games with puzzle...,GameTool,True
1,Why is it so difficult to find a rental proper...,HouseRentingTool,True
2,Any suggestions for quick and easy dinner ideas?,DietTool,True
3,What is the formula for the volume of a sphere?,,False
4,What does the word 'ephemeral' mean?,,False


In [8]:
# Sanity check: per-tool counts and balance.
print('total:', len(sampled))
print('positive:', int(sampled['requires_tool'].sum()))
print('negative:', int((~sampled['requires_tool']).sum()))
print('\nper-tool counts (positives only):')
print(sampled[sampled['requires_tool']]['gold_tool'].value_counts().describe())

total: 470
positive: 235
negative: 235

per-tool counts (positives only):
count    47.0
mean      5.0
std       0.0
min       5.0
25%       5.0
50%       5.0
75%       5.0
max       5.0
Name: count, dtype: float64


## 4. Render the prompt for one query

Visual sanity check: the same prompt is sent to all three models (only the chat template wrapper differs).

In [9]:
from src.prompts import render_prompt
from src.sampling import load_tool_list
tools = load_tool_list(Path('data/big_tool_des.json'))
print(render_prompt(sampled.iloc[0]['query'], tools)[:2000])

You are a tool-routing assistant. For each user query you do TWO things:
  1. Tool Usage Awareness: decide whether the query requires using one of the listed tools.
     A query requires a tool only if it cannot be answered from general knowledge or simple
     reasoning alone — for example, when it needs real-time data, an API call, code execution,
     image generation, or external services.
  2. Tool Selection: if a tool is required, pick the SINGLE best matching tool from the list
     by its exact name.

Available tools (you must use one of these exact names if a tool is needed):
  1. FinanceTool: Stay informed with the latest financial updates, real-time insights, and analysis on a wide range of options, stocks, cryptocurrencies, and more.
  2. ExchangeTool: Seamlessly convert currencies with our integrated currency conversion tool.
  3. NewsTool: Stay connected to global events with our up-to-date news around the world.
  4. PolishTool: Elevate your content with our AI-powered t

In [10]:
from huggingface_hub import whoami
whoami()

{'type': 'user',
 'id': '69115385ea7fa18ab6911fbd',
 'name': 'jamelski',
 'fullname': 'Jamel Lawson',
 'email': 'lawson.l.jamel@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1777593600,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/A8j-NkWQV66odadx-dpHs.png',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Homework 2 Read',
   'role': 'read',
   'createdAt': '2026-04-25T04:48:35.458Z'}}}

In [11]:
import subprocess
r = subprocess.run(['git', '-C', '/content/hold-the-line', 'pull'],
                   capture_output=True, text=True)
print('pull:', r.stdout, r.stderr)

r = subprocess.run(['git', '-C', '/content/hold-the-line', 'log', '-1', '--oneline'],
                   capture_output=True, text=True)
print('HEAD:', r.stdout)

# Show the actual line that's failing — should now mention return_dict=True.
!grep -n -A 3 'apply_chat_template' /content/hold-the-line/metatool-agent/src/llm_clients.py

pull: Already up to date.
 
HEAD: b83fd47 Fix BatchEncoding handling in LLMClient.generate

92:        # apply_chat_template returns a bare tensor on older transformers and a
93-        # BatchEncoding on newer versions. Forcing return_dict=True normalizes
94-        # this so we always get attention_mask alongside input_ids.
95:        encoded = self.tokenizer.apply_chat_template(
96-            messages,
97-            add_generation_prompt=True,
98-            return_tensors="pt",


## 5. Run all three models sequentially

Each model is loaded, run over the full sample, and unloaded before the next is loaded. Predictions and malformed-response logs are written to `results/`.

In [ ]:
from pathlib import Path
import pandas as pd
from src.agent import run_agent
from src.evaluate import evaluate_predictions, write_summary
from src.llm_clients import ALL_SPECS, build_client
from src.sampling import load_tool_list

tools = load_tool_list(Path('data/big_tool_des.json'))
sampled = pd.read_csv('data/sampled_data.csv')
results_dir = Path('results'); results_dir.mkdir(exist_ok=True)

metrics_all = []
for spec in ALL_SPECS:
    print(f'\n=== {spec.label} ===')
    client = build_client(spec)
    preds_path     = results_dir / f'predictions_{spec.label}.csv'
    malformed_path = results_dir / f'malformed_{spec.label}.jsonl'
    metrics_path   = results_dir / f'metrics_{spec.label}.json'

    run_agent(client, sampled, tools,
              out_predictions_path=preds_path,
              out_malformed_path=malformed_path,
              max_tokens=256)
    metrics_all.append(evaluate_predictions(preds_path, metrics_path, spec.label))

    client.unload(); del client

summary = write_summary(metrics_all, results_dir / 'summary.csv')
summary


=== gemma2-2b-it ===


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## 6. Inspect a few representative failures per model

Use this to populate the error-analysis section of `report.md`.

In [ ]:
import pandas as pd
for spec in ALL_SPECS:
    p = f'results/predictions_{spec.label}.csv'
    df = pd.read_csv(p)
    print(f'\n=== {spec.label} — failures ===')
    df['pred_tool_needed_filled'] = df['pred_tool_needed'].fillna(False).astype(bool)
    wrong_binary = df[df['requires_tool'] != df['pred_tool_needed_filled']]
    wrong_tool = df[df['requires_tool'] & df['pred_tool_needed_filled'] &
                    (df['gold_tool'].astype(str).str.strip() != df['pred_tool_name'].fillna('').astype(str).str.strip())]
    malformed = df[df['parse_status'] == 'malformed']
    print(f'wrong-binary: {len(wrong_binary)} | wrong-tool: {len(wrong_tool)} | malformed: {len(malformed)}')
    if len(wrong_tool):
        print(wrong_tool[['query','gold_tool','pred_tool_name']].head(3).to_string())